# Business Insight

Cluster analysis shows that different customer groups have different subscription rates.

This confirms that customer segmentation provides valuable information that can improve marketing campaign targeting.

The cluster label will therefore be used as an additional feature during predictive modeling.

Imports

In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

import joblib

Load Dataset

In [2]:
df = pd.read_csv("../data/processed/clustered_data.csv")

print(df.shape)

df.head()

(41188, 26)


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,cons.conf.idx,euribor3m,nr.employed,y,previous_contact,multiple_contacts,is_senior,stable_job,summer_contact,cluster
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,-36.4,4.857,5191.0,no,0,0,0,0,1,1
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,-36.4,4.857,5191.0,no,0,0,0,0,1,1
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,-36.4,4.857,5191.0,no,0,0,0,0,1,1
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,-36.4,4.857,5191.0,no,0,0,0,1,1,1
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,-36.4,4.857,5191.0,no,0,0,0,0,1,1


Target Encoding

In [3]:
df["y"] = df["y"].map({
    "no":0,
    "yes":1
})

df["y"].value_counts()

y
0    36548
1     4640
Name: count, dtype: int64

Features & Target

In [4]:
X = df.drop("y", axis=1)

y = df["y"]

Numerical & Categorical Features

In [5]:
categorical_features = X.select_dtypes(include="object").columns.tolist()

numerical_features = X.select_dtypes(exclude="object").columns.tolist()

print("Categorical:", len(categorical_features))

print("Numerical:", len(numerical_features))

Categorical: 10
Numerical: 15


Train Test Split

In [6]:
X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.20,

    stratify=y,

    random_state=42
)

print(X_train.shape)

print(X_test.shape)

(32950, 25)
(8238, 25)


Preprocessing Pipeline

In [7]:
preprocessor = ColumnTransformer(

    transformers=[

        (

            "num",

            StandardScaler(),

            numerical_features

        ),

        (

            "cat",

            OneHotEncoder(

                handle_unknown="ignore"

            ),

            categorical_features

        )

    ]
)

Save Pipeline

In [8]:
joblib.dump(

    preprocessor,

    "../models/preprocessor.pkl"

)

print("Pipeline Saved.")

Pipeline Saved.


Fit Pipeline

In [9]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

print(X_train_processed.shape)

print(X_test_processed.shape)

(32950, 68)
(8238, 68)


Logistic Regression Pipeline

In [10]:
logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        random_state=42,
        max_iter=1000
    ))
])

logistic_model.fit(X_train, y_train)

print("Logistic Regression Trained Successfully.")

Logistic Regression Trained Successfully.


Logistic Predictions

In [11]:
y_pred_lr = logistic_model.predict(X_test)

y_prob_lr = logistic_model.predict_proba(X_test)[:,1]

Logistic Evaluation

In [12]:
lr_results = {
    "Model": "Logistic Regression",

    "Accuracy (%)": accuracy_score(y_test, y_pred_lr) * 100,

    "Precision (%)": precision_score(y_test, y_pred_lr) * 100,

    "Recall (%)": recall_score(y_test, y_pred_lr) * 100,

    "F1 Score (%)": f1_score(y_test, y_pred_lr) * 100,

    "ROC AUC (%)": roc_auc_score(y_test, y_prob_lr) * 100
}

pd.DataFrame([lr_results]).round(2)

,Model,Accuracy (%),Precision (%),Recall (%),F1 Score (%),ROC AUC (%)
0,Logistic Regression,90.12,68.87,22.41,33.82,80.09


Classification Report

In [13]:
print(classification_report(y_test, y_pred_lr))

              precision    recall  f1-score   support

           0       0.91      0.99      0.95      7310
           1       0.69      0.22      0.34       928

    accuracy                           0.90      8238
   macro avg       0.80      0.61      0.64      8238
weighted avg       0.88      0.90      0.88      8238



Random Forest Pipeline

In [14]:
rf_model = Pipeline([
    ("preprocessor", preprocessor),

    ("classifier",
     RandomForestClassifier(

         n_estimators=300,

         random_state=42,

         n_jobs=-1
     ))
])

rf_model.fit(X_train, y_train)

print("Random Forest Trained Successfully.")

Random Forest Trained Successfully.


Random Forest Predictions

In [15]:
y_pred_rf = rf_model.predict(X_test)

y_prob_rf = rf_model.predict_proba(X_test)[:,1]

Random Forest Evaluation

In [16]:
rf_results = {

    "Model": "Random Forest",

    "Accuracy (%)": accuracy_score(y_test, y_pred_rf) * 100,

    "Precision (%)": precision_score(y_test, y_pred_rf) * 100,

    "Recall (%)": recall_score(y_test, y_pred_rf) * 100,

    "F1 Score (%)": f1_score(y_test, y_pred_rf) * 100,

    "ROC AUC (%)": roc_auc_score(y_test, y_prob_rf) * 100
}

pd.DataFrame([rf_results]).round(2)

,Model,Accuracy (%),Precision (%),Recall (%),F1 Score (%),ROC AUC (%)
0,Random Forest,89.66,58.05,29.53,39.14,78.63


Classification Report

In [17]:
print(classification_report(y_test, y_pred_rf))

              precision    recall  f1-score   support

           0       0.92      0.97      0.94      7310
           1       0.58      0.30      0.39       928

    accuracy                           0.90      8238
   macro avg       0.75      0.63      0.67      8238
weighted avg       0.88      0.90      0.88      8238



Compare Both Models

In [18]:
comparison = pd.DataFrame([
    lr_results,
    rf_results
])

comparison = comparison.round(2)

comparison

,Model,Accuracy (%),Precision (%),Recall (%),F1 Score (%),ROC AUC (%)
0,Logistic Regression,90.12,68.87,22.41,33.82,80.09
1,Random Forest,89.66,58.05,29.53,39.14,78.63


Save Comparison

In [19]:
comparison.to_csv(
    "../reports/baseline_model_comparison.csv",
    index=False
)

print("Baseline comparison saved successfully.")

Baseline comparison saved successfully.


Save Models

In [20]:
joblib.dump(
    logistic_model,
    "../models/logistic_regression.pkl"
)

joblib.dump(
    rf_model,
    "../models/random_forest.pkl"
)

print("Models saved successfully.")

Models saved successfully.


Best Baseline Model

In [21]:
best_baseline = comparison.sort_values(
    by="ROC AUC (%)",
    ascending=False
)

best_baseline

,Model,Accuracy (%),Precision (%),Recall (%),F1 Score (%),ROC AUC (%)
0,Logistic Regression,90.12,68.87,22.41,33.82,80.09
1,Random Forest,89.66,58.05,29.53,39.14,78.63


# Baseline Model Summary

Two baseline models were trained:

- Logistic Regression
- Random Forest

Random Forest is expected to outperform Logistic Regression due to its ability to model non-linear relationships.

More advanced models such as XGBoost, LightGBM and CatBoost will be developed in the next notebook.